#file handling

In [9]:
# ============================================================
#       NDMU CLASS GRADE BOOK
#       DARK GLOW WEB PORTAL
#       GOOGLE COLAB VERSION  (multi-student)
# ============================================================

!pip -q install openpyxl reportlab ipywidgets

# ============================================================
# IMPORTS
# ============================================================

from google.colab import files, output
output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
import html
import re

from IPython.display import display, HTML, clear_output

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.platypus import (
    SimpleDocTemplate, Table, TableStyle,
    Paragraph, Spacer, CondPageBreak
)

# ============================================================
# STUDENT INFORMATION
# ============================================================

print("==============================================")
print("       NDMU CLASS GRADE BOOK")
print("==============================================")
print()

student_name = input("👤 Enter Student Name: ").strip() or "Student"
course = input("🎓 Enter Course: ").strip() or "Not Specified"
year_level = input("📚 Enter Year Level: ").strip() or "Not Specified"

print()
print("Student   :", student_name)
print("Course    :", course)
print("Year Level:", year_level)
print()

# ============================================================
# UPLOAD EXCEL FILE
# ============================================================

print("📁 Please upload the Excel file (.xlsx)")
print()

uploaded = files.upload()

if not uploaded:
    raise Exception("No Excel file was uploaded.")

file_name = list(uploaded.keys())[0]

print()
print("✅ File uploaded:", file_name)

# ============================================================
# READ EXCEL
# ============================================================

raw = pd.read_excel(file_name)

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

raw.columns = raw.columns.astype(str).str.strip().str.upper()

# Accept PROFESSOR as the professor/teacher column
if "TEACHER" not in raw.columns and "PROFESSOR" in raw.columns:
    raw.rename(columns={"PROFESSOR": "TEACHER"}, inplace=True)

required_columns = ["CODE", "DESCRIPTION", "TEACHER", "UNITS", "GRADE"]

missing_columns = [
    column for column in required_columns
    if column not in raw.columns
]

if missing_columns:
    print("❌ Missing columns:", missing_columns)
    print("Your Excel columns are:", list(raw.columns))
    raise Exception(
        "Please make sure your Excel file contains "
        "CODE, DESCRIPTION, PROFESSOR/TEACHER, UNITS, and GRADE."
    )

# ============================================================
# CLEAN DATA
# ============================================================

for column in ["CODE", "DESCRIPTION", "TEACHER"]:
    raw[column] = raw[column].fillna("").astype(str).str.strip()

raw["UNITS"] = pd.to_numeric(raw["UNITS"], errors="coerce")
raw["GRADE"] = pd.to_numeric(raw["GRADE"], errors="coerce")

raw = raw.reset_index(drop=True)

# ============================================================
# SPLIT THE SHEET INTO STUDENTS
#
#  - A NAME ROW has exactly ONE text cell (the student's name)
#    and no units or grade.  It starts a new student.
#  - A BLANK ROW is skipped.
#  - Every other row is a subject of the current student.
# ============================================================

records = []
current_student = "Student"

for _, row in raw.iterrows():
    texts = [row["CODE"], row["DESCRIPTION"], row["TEACHER"]]
    filled = [text for text in texts if text != ""]
    no_numbers = pd.isna(row["UNITS"]) and pd.isna(row["GRADE"])

    if not filled and no_numbers:
        continue                      # blank separator row

    if len(filled) == 1 and no_numbers:
        current_student = filled[0]   # student name row
        continue

    record = {column: row[column] for column in required_columns}
    record["STUDENT"] = current_student
    records.append(record)

if not records:
    raise Exception("No subject records were found in the Excel file.")

df = pd.DataFrame(records).reset_index(drop=True)

student_list = list(df["STUDENT"].unique())

print()
print(f"👥 Students found : {len(student_list)}")
print(f"📚 Subject records: {len(df)}")

# ============================================================
# HIDE THE SETUP MESSAGES (they are still printed, then cleared)
# Set SHOW_SETUP_MESSAGES = True to keep them visible.
# ============================================================

SHOW_SETUP_MESSAGES = False

if not SHOW_SETUP_MESSAGES:
    clear_output()

# ============================================================
# GRADE HELPERS
# On the Filipino grading scale, 1.00 is the BEST grade and
# 5.00 is the WORST grade, so:
#   HIGHEST grade = numerically smallest value
#   LOWEST grade  = numerically largest value
# ============================================================

def grade_stats(data):
    grades = data["GRADE"].dropna()
    total = len(data)
    average = grades.mean() if not grades.empty else 0
    lowest = grades.max() if not grades.empty else 0    # worst grade
    highest = grades.min() if not grades.empty else 0   # best grade
    return total, average, lowest, highest

def format_units(value):
    if pd.isna(value):
        return ""
    return f"{float(value):g}"

def format_grade(value):
    if pd.isna(value):
        return "N/A"
    return f"{float(value):.2f}"

# ============================================================
# DISPLAY HELPERS  (names, initials, grade colors)
# ============================================================

def split_student(label):
    """'JOHN DELA CRUZ - ENGINEERING' -> ('JOHN DELA CRUZ', 'ENGINEERING')"""
    label = str(label).strip()

    if " - " in label:
        name, extra = label.split(" - ", 1)
        return name.strip(), extra.strip()

    return label, ""

def initials(name):
    parts = [part for part in re.split(r"\s+", str(name).strip()) if part]

    if not parts:
        return "?"

    if len(parts) == 1:
        return parts[0][:2].upper()

    return (parts[0][0] + parts[-1][0]).upper()

def grade_class(value):
    if pd.isna(value):
        return "grade-na"

    value = float(value)

    if value <= 1.50:
        return "grade-excellent"
    if value <= 2.25:
        return "grade-good"
    if value <= 3.00:
        return "grade-fair"

    return "grade-fail"

def grade_pill(value):
    return f'<span class="grade-pill {grade_class(value)}">{format_grade(value)}</span>'

# ============================================================
# CSS - DARK GLOW DESIGN
# ============================================================

display(HTML("""
<style>
@keyframes ndmuSpin { to { transform: rotate(360deg); } }
@keyframes ndmuShimmer {
    0%   { background-position: -200% 0; }
    100% { background-position: 200% 0; }
}
@keyframes ndmuPulse {
    0%, 100% { box-shadow: 0 0 10px rgba(255,215,50,.45); }
    50%      { box-shadow: 0 0 28px rgba(255,215,50,.95); }
}
@keyframes ndmuBlink { 0%, 100% { opacity: 1; } 50% { opacity: .25; } }
@keyframes ndmuFadeUp {
    from { opacity: 0; transform: translateY(14px); }
    to   { opacity: 1; transform: none; }
}
@keyframes ndmuFloat {
    0%, 100% { transform: translateY(0) rotate(-6deg); }
    50%      { transform: translateY(-7px) rotate(8deg); }
}

body {
    background:
        radial-gradient(circle at 50% 0%, rgba(0,255,120,.15), transparent 35%),
        radial-gradient(circle at 0% 100%, rgba(0,180,90,.10), transparent 35%),
        #020706 !important;
    font-family: Arial, sans-serif !important;
}

/* ---------------- HEADER ---------------- */

.ndmu-header {
    position: relative;
    overflow: hidden;
    padding: 28px 32px 24px 32px;
    border-radius: 20px;
    color: white;
    background:
        radial-gradient(circle at 92% 8%, rgba(255,215,64,.30), transparent 40%),
        radial-gradient(circle at 8% 115%, rgba(0,255,140,.25), transparent 45%),
        repeating-linear-gradient(135deg, rgba(255,255,255,.035) 0 2px, transparent 2px 14px),
        linear-gradient(135deg, #002e1b, #00683a 55%, #009150);
    border: 1px solid rgba(255,215,70,.45);
    box-shadow: 0 0 22px rgba(0,255,120,.30),
                0 0 60px rgba(255,215,64,.10),
                0 14px 40px rgba(0,0,0,.60);
}

.ndmu-header::before {
    content: "";
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 4px;
    background: linear-gradient(90deg, transparent, #ffd740, #00e676, #ffd740, transparent);
    background-size: 200% 100%;
    animation: ndmuShimmer 4s linear infinite;
}

.header-row {
    display: flex;
    align-items: center;
    justify-content: space-between;
    flex-wrap: wrap;
    gap: 14px;
}

.logo-wrap {
    position: relative;
    width: 86px;
    height: 86px;
    flex-shrink: 0;
}

.logo-ring {
    position: absolute;
    top: 0; left: 0; right: 0; bottom: 0;
    border-radius: 50%;
    background: conic-gradient(#ffd740, #00e676, #ffffff, #ffd740);
    box-shadow: 0 0 24px rgba(255,215,64,.65);
    animation: ndmuSpin 6s linear infinite;
}

.logo {
    position: absolute;
    top: 5px; left: 5px; right: 5px; bottom: 5px;
    border-radius: 50%;
    background: radial-gradient(circle at 30% 25%, #fff3a8, #ffd23f 45%, #d99a00);
    color: #00532e;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    font-size: 17px;
    font-weight: 900;
    letter-spacing: 1px;
    line-height: 1.1;
}

.logo small { font-size: 8px; letter-spacing: 2px; margin-top: 2px; }

.school-name {
    font-size: 25px;
    font-weight: 800;
    letter-spacing: 1.5px;
    background: linear-gradient(90deg, #ffffff, #fff3b0 60%, #ffd740);
    -webkit-background-clip: text;
    background-clip: text;
    -webkit-text-fill-color: transparent;
    filter: drop-shadow(0 0 8px rgba(255,215,64,.35));
}

.school-subtitle {
    font-size: 11px;
    color: #c9f5dc;
    margin-top: 5px;
    letter-spacing: 1px;
}

.portal-title {
    margin-top: 24px;
    font-size: 44px;
    font-weight: 900;
    letter-spacing: 3px;
    line-height: 1.05;
    background: linear-gradient(90deg, #ffffff 30%, #ffe45c);
    -webkit-background-clip: text;
    background-clip: text;
    -webkit-text-fill-color: transparent;
    filter: drop-shadow(0 0 12px rgba(255,215,64,.30));
}

.portal-subtitle {
    margin-top: 8px;
    font-size: 12px;
    letter-spacing: 4px;
    text-transform: uppercase;
    color: #c9f5dc;
    display: flex;
    align-items: center;
    gap: 10px;
}

.portal-subtitle::before {
    content: "";
    width: 40px;
    height: 2px;
    background: linear-gradient(90deg, #ffd740, transparent);
}

.header-chips {
    display: flex;
    gap: 10px;
    margin-top: 18px;
    flex-wrap: wrap;
}

.header-chip {
    background: rgba(0,0,0,.30);
    border: 1px solid rgba(255,215,64,.35);
    padding: 7px 15px;
    border-radius: 999px;
    font-size: 11px;
    font-weight: 700;
    color: #fff3b0;
}

.portal-badge {
    display: flex;
    align-items: center;
    gap: 8px;
    background: linear-gradient(135deg, #ffe86b, #e5ae00);
    color: #00351f;
    padding: 11px 22px;
    border-radius: 999px;
    font-size: 11px;
    font-weight: 800;
    letter-spacing: 1.5px;
    animation: ndmuPulse 2.5s ease-in-out infinite;
}

.live-dot {
    width: 9px;
    height: 9px;
    border-radius: 50%;
    background: #00a84f;
    box-shadow: 0 0 8px #00e676;
    animation: ndmuBlink 1.4s infinite;
}

.glow-line {
    height: 2px;
    margin: 14px 0;
    background: linear-gradient(90deg, transparent, #00e676, #ffd740, #00e676, transparent);
    box-shadow: 0 0 12px #00e676;
}

/* ---------------- NAVIGATION ---------------- */

.ndmu-nav {
    border-radius: 999px !important;
    background: linear-gradient(135deg, rgba(0,107,63,.65), rgba(0,55,32,.85)) !important;
    border: 1px solid rgba(0,255,120,.28) !important;
    color: #cdeedd !important;
    font-weight: 800 !important;
    font-size: 12px !important;
    letter-spacing: 1px !important;
    box-shadow: 0 0 10px rgba(0,255,120,.10) !important;
    transition: all .25s ease !important;
    cursor: pointer;
}

.ndmu-nav:hover {
    transform: translateY(-2px);
    color: white !important;
    border-color: #00e676 !important;
    box-shadow: 0 0 20px rgba(0,255,120,.50) !important;
}

.ndmu-nav-active {
    background: linear-gradient(135deg, #ffe86b, #e5ae00) !important;
    color: #00351f !important;
    border-color: #ffe86b !important;
    box-shadow: 0 0 22px rgba(255,215,64,.60) !important;
}

/* ---------------- INPUTS + BUTTONS ---------------- */

.ndmu-input select, .ndmu-input input {
    background: rgba(2,18,11,.95) !important;
    color: #dff5e8 !important;
    border: 1px solid rgba(0,255,120,.35) !important;
    border-radius: 12px !important;
    padding: 0 14px !important;
    font-size: 12px !important;
    height: 40px !important;
    box-shadow: inset 0 0 12px rgba(0,255,120,.06) !important;
}

.ndmu-input input::placeholder { color: #6f9482 !important; }

.ndmu-input select:focus, .ndmu-input input:focus {
    outline: none !important;
    border-color: #ffd740 !important;
    box-shadow: 0 0 14px rgba(255,215,64,.45) !important;
}

.ndmu-input option { background: #06180f; color: #dff5e8; }

.ndmu-btn {
    border-radius: 12px !important;
    border: none !important;
    font-weight: 800 !important;
    font-size: 12px !important;
    letter-spacing: 1px !important;
    transition: all .25s ease !important;
    cursor: pointer;
}

.ndmu-btn:hover { transform: translateY(-2px); filter: brightness(1.15); }

.ndmu-btn-search {
    background: linear-gradient(135deg, #00a85a, #00e676) !important;
    color: #00281a !important;
    box-shadow: 0 0 14px rgba(0,230,118,.45) !important;
}

.ndmu-btn-clear {
    background: linear-gradient(135deg, #3d4e45, #26332d) !important;
    color: #e6f3ec !important;
    border: 1px solid rgba(255,255,255,.12) !important;
}

.ndmu-btn-pdf {
    background: linear-gradient(135deg, #ffe86b, #e5ae00) !important;
    color: #3a2a00 !important;
    box-shadow: 0 0 14px rgba(255,215,64,.45) !important;
}

.ndmu-panel {
    box-sizing: border-box !important;
    background: linear-gradient(145deg, #10251a, #07140e);
    padding: 17px !important;
    border-radius: 13px;
    margin-bottom: 14px;
    border: 1px solid rgba(0,255,120,.18);
    box-shadow: 0 0 12px rgba(0,255,120,.10),
                0 10px 30px rgba(0,0,0,.55);
}

/* ---------------- STUDENT PILLS ---------------- */

@keyframes ndmuGlow {
    0%, 100% { box-shadow: 0 0 14px rgba(0,230,118,.55); }
    50%      { box-shadow: 0 0 30px rgba(0,230,118,.95); }
}

.filter-label {
    color: #82a996;
    font-size: 9px;
    letter-spacing: 2px;
    margin: 2px 0 8px 2px;
}

.ndmu-pill {
    border-radius: 999px !important;
    background: rgba(0,180,90,.12) !important;
    border: 1px solid rgba(0,255,120,.30) !important;
    color: #bfe9d1 !important;
    font-weight: 800 !important;
    font-size: 11px !important;
    letter-spacing: .5px !important;
    padding: 0 18px !important;
    transition: all .25s ease !important;
    cursor: pointer;
    animation: ndmuFadeUp .5s ease both;
}

.ndmu-pill:hover {
    transform: translateY(-3px) scale(1.03);
    border-color: #ffd740 !important;
    color: white !important;
    box-shadow: 0 0 18px rgba(255,215,64,.45) !important;
}

.ndmu-pill-active {
    background: linear-gradient(135deg, #00c26a, #00e676) !important;
    color: #00281a !important;
    border-color: #b6ffd8 !important;
    animation: ndmuGlow 2s ease-in-out infinite !important;
}

.result-chip {
    display: inline-block;
    margin-top: 12px;
    background: rgba(255,215,64,.10);
    border: 1px solid rgba(255,215,64,.30);
    color: #ffe27a;
    border-radius: 999px;
    padding: 6px 14px;
    font-size: 11px;
}

.result-chip b { color: #fff3b0; }

/* ---------------- CARDS ---------------- */

.card {
    background: linear-gradient(145deg, #10251a, #07140e);
    padding: 17px;
    border-radius: 13px;
    margin-bottom: 14px;
    border: 1px solid rgba(0,255,120,.18);
    box-shadow: 0 0 12px rgba(0,255,120,.10),
                0 10px 30px rgba(0,0,0,.55);
    animation: ndmuFadeUp .55s ease both;
    transition: border-color .3s ease, box-shadow .3s ease;
}

.card:hover {
    border-color: rgba(0,255,120,.42);
    box-shadow: 0 0 22px rgba(0,255,120,.22),
                0 10px 30px rgba(0,0,0,.55);
}

.card-title {
    display: flex;
    align-items: center;
    gap: 10px;
    color: #58e99c;
    font-size: 17px;
    font-weight: 800;
    margin-bottom: 12px;
}

.card-title::before {
    content: "";
    width: 5px;
    height: 20px;
    border-radius: 3px;
    background: linear-gradient(#ffd740, #00e676);
    box-shadow: 0 0 10px rgba(0,230,118,.60);
}

/* ---------------- WELCOME ---------------- */

.welcome {
    position: relative;
    overflow: hidden;
    background:
        radial-gradient(circle at 95% 0%, rgba(255,215,64,.25), transparent 40%),
        linear-gradient(135deg, #005f37, #00905a);
    color: white;
    padding: 22px 24px;
    border-radius: 16px;
    margin-bottom: 14px;
    border: 1px solid rgba(255,215,70,.25);
    box-shadow: 0 0 18px rgba(0,255,120,.25),
                0 8px 25px rgba(0,0,0,.45);
    animation: ndmuFadeUp .55s ease both;
}

.welcome-top { display: flex; align-items: center; gap: 16px; }

.welcome-wave {
    font-size: 38px;
    animation: ndmuFloat 3s ease-in-out infinite;
}

.welcome-title { font-size: 22px; font-weight: 800; letter-spacing: .5px; }
.welcome-text  { font-size: 12px; color: #d7eee1; margin-top: 5px; }

.feature-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(190px, 1fr));
    gap: 12px;
    margin-top: 18px;
}

.feature {
    background: rgba(0,0,0,.22);
    border: 1px solid rgba(255,255,255,.12);
    border-radius: 12px;
    padding: 13px 15px;
    transition: all .25s ease;
}

.feature:hover {
    transform: translateY(-3px);
    background: rgba(0,0,0,.32);
    border-color: rgba(255,215,64,.55);
    box-shadow: 0 0 16px rgba(255,215,64,.25);
}

.feature-icon { font-size: 22px; }
.feature-name { font-size: 13px; font-weight: 800; margin-top: 6px; color: #fff3b0; }
.feature-desc { font-size: 10px; color: #cfe9da; margin-top: 3px; line-height: 1.4; }

/* ---------------- WELCOME PANEL (clickable tiles) ---------------- */

.welcome-panel {
    box-sizing: border-box !important;
    position: relative;
    overflow: hidden;
    background:
        radial-gradient(circle at 95% 0%, rgba(255,215,64,.25), transparent 40%),
        linear-gradient(135deg, #005f37, #00905a);
    color: white;
    padding: 22px 24px !important;
    border-radius: 16px;
    margin-bottom: 14px;
    border: 1px solid rgba(255,215,70,.25);
    box-shadow: 0 0 18px rgba(0,255,120,.25),
                0 8px 25px rgba(0,0,0,.45);
    animation: ndmuFadeUp .55s ease both;
}

.ndmu-feature {
    background: rgba(0,0,0,.24) !important;
    border: 1px solid rgba(255,255,255,.14) !important;
    border-radius: 12px !important;
    color: #fff3b0 !important;
    font-weight: 700 !important;
    font-size: 12px !important;
    line-height: 1.6 !important;
    text-align: left !important;
    justify-content: flex-start !important;
    white-space: pre-line !important;
    padding: 12px 16px !important;
    transition: all .25s ease !important;
    cursor: pointer;
}

.ndmu-feature:hover {
    transform: translateY(-4px);
    background: rgba(0,0,0,.36) !important;
    border-color: rgba(255,215,64,.70) !important;
    color: white !important;
    box-shadow: 0 0 20px rgba(255,215,64,.40) !important;
}

.ndmu-feature:active { transform: translateY(-1px) scale(.98); }

/* ---------------- STUDENT INFO ---------------- */

.student-card {
    display: flex;
    align-items: center;
    gap: 18px;
    background: linear-gradient(145deg, #10281c, #07150e);
    padding: 16px 20px;
    border-radius: 14px;
    margin-bottom: 14px;
    border: 1px solid rgba(0,255,120,.18);
    box-shadow: 0 0 12px rgba(0,255,120,.10),
                0 10px 25px rgba(0,0,0,.45);
    animation: ndmuFadeUp .55s ease both;
}

.avatar {
    width: 58px;
    height: 58px;
    border-radius: 50%;
    flex-shrink: 0;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 19px;
    font-weight: 900;
    color: #00351f;
    background: linear-gradient(145deg, #fff3a8, #ffd23f 55%, #d99a00);
    border: 3px solid rgba(255,255,255,.85);
    box-shadow: 0 0 18px rgba(255,215,64,.55);
}

.avatar-sm { width: 44px; height: 44px; font-size: 15px; border-width: 2px; }
.avatar-xs { width: 32px; height: 32px; font-size: 11px; border-width: 2px; box-shadow: 0 0 10px rgba(255,215,64,.45); }

.student-label { color: #82a996; font-size: 9px; letter-spacing: 2px; text-transform: uppercase; }
.student-name  { color: #62f0a4; font-size: 20px; font-weight: 800; margin-top: 3px; }

.student-details { display: flex; gap: 10px; margin-top: 9px; flex-wrap: wrap; }

.info-pill {
    background: rgba(0,180,90,.12);
    border: 1px solid rgba(0,255,120,.25);
    color: #b8d4c5;
    border-radius: 999px;
    padding: 5px 13px;
    font-size: 11px;
}

.info-pill b { color: #62f0a4; }

/* ---------------- STUDENT NAME CHIPS ---------------- */

.name-chip {
    display: inline-flex;
    align-items: center;
    gap: 9px;
    background: rgba(0,180,90,.12);
    border: 1px solid rgba(0,255,120,.25);
    border-radius: 999px;
    padding: 6px 14px 6px 6px;
    margin: 4px 8px 4px 0;
    transition: all .25s ease;
}

.name-chip:hover {
    transform: translateY(-3px);
    border-color: #ffd740;
    box-shadow: 0 0 16px rgba(255,215,64,.35);
}

.chip-num {
    width: 24px;
    height: 24px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    background: linear-gradient(145deg, #ffe86b, #e5ae00);
    color: #00351f;
    font-size: 11px;
    font-weight: 900;
}

.chip-name   { color: #62f0a4; font-size: 11px; font-weight: 800; }
.chip-course { color: #ffe27a; font-size: 10px; font-weight: 700; opacity: .9; }

/* ---------------- STAT TILES ---------------- */

.stat-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(190px, 1fr));
    gap: 12px;
}

.stat {
    display: flex;
    align-items: center;
    gap: 13px;
    background: linear-gradient(145deg, #142b20, #0a1811);
    border-left: 4px solid var(--accent);
    padding: 13px 14px;
    border-radius: 10px;
    box-shadow: inset 0 0 14px rgba(0,255,120,.05);
    transition: all .25s ease;
}

.stat:hover {
    transform: translateY(-3px);
    box-shadow: 0 0 18px rgba(0,255,120,.20), inset 0 0 14px rgba(0,255,120,.06);
}

.stat-icon {
    width: 42px;
    height: 42px;
    border-radius: 12px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 21px;
    background: rgba(255,255,255,.06);
    border: 1px solid rgba(255,255,255,.10);
}

.stat-number { color: var(--accent); font-size: 26px; font-weight: 900; text-shadow: 0 0 14px var(--accent); }
.stat-label  { color: #8da99b; font-size: 9px; letter-spacing: 1.5px; margin-top: 3px; }

/* ---------------- TABLES ---------------- */

.table-wrap {
    border-radius: 12px;
    overflow-x: auto;
    border: 1px solid rgba(0,255,120,.15);
}

.grade-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 11px;
    color: #dcebe3;
}

.grade-table th {
    background: linear-gradient(135deg, #005c35, #00814a);
    color: white;
    padding: 11px 10px;
    text-align: left;
    font-size: 10px;
    letter-spacing: 1.5px;
}

.grade-table td {
    padding: 10px;
    border-bottom: 1px solid rgba(255,255,255,.07);
    color: #cad9d1;
    vertical-align: middle;
}

.grade-table tr:nth-child(even) { background: rgba(255,255,255,.025); }
.grade-table tr:hover { background: rgba(0,255,120,.09); }

.desc-cell { font-weight: 700; color: #e6f3ec !important; }
.prof-cell { color: #9fbbaa !important; }

.row-num {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    width: 22px;
    height: 22px;
    border-radius: 50%;
    background: rgba(0,255,120,.10);
    color: #62f0a4;
    font-size: 10px;
    font-weight: 800;
}

.code-tag {
    display: inline-block;
    background: rgba(255,215,64,.12);
    border: 1px solid rgba(255,215,64,.35);
    color: #ffe27a;
    border-radius: 7px;
    padding: 3px 9px;
    font-weight: 800;
    letter-spacing: .5px;
}

.grade-pill {
    display: inline-block;
    min-width: 40px;
    text-align: center;
    padding: 4px 11px;
    border-radius: 999px;
    font-weight: 800;
    font-size: 11px;
    border: 1px solid;
}

.grade-excellent { background: rgba(0,230,118,.16); color: #5af0a2; border-color: rgba(0,255,120,.45); box-shadow: 0 0 10px rgba(0,255,120,.22); }
.grade-good      { background: rgba(190,240,90,.13); color: #d2f57a; border-color: rgba(190,240,90,.40); }
.grade-fair      { background: rgba(255,193,7,.14);  color: #ffd54f; border-color: rgba(255,193,7,.42); }
.grade-fail      { background: rgba(255,82,82,.16);  color: #ff8a80; border-color: rgba(255,82,82,.48); }
.grade-na        { background: rgba(255,255,255,.06); color: #9fb5a8; border-color: rgba(255,255,255,.16); }

/* ---------------- STUDENT BLOCK ---------------- */

.student-block-header {
    display: flex;
    align-items: center;
    gap: 14px;
    flex-wrap: wrap;
    background:
        radial-gradient(circle at 100% 0%, rgba(255,215,64,.22), transparent 45%),
        linear-gradient(135deg, #006b3f, #00a85a);
    padding: 12px 16px;
    border-radius: 12px;
    margin-bottom: 14px;
    border: 1px solid rgba(255,255,255,.12);
    box-shadow: 0 0 16px rgba(0,255,120,.28);
}

.sb-names { flex: 1; min-width: 180px; }
.sb-name   { color: white; font-size: 16px; font-weight: 900; letter-spacing: 1px; }
.sb-course { color: #fff3b0; font-size: 10px; font-weight: 800; letter-spacing: 2px; margin-top: 2px; }

.sb-pills { display: flex; gap: 8px; flex-wrap: wrap; }

.sb-pill {
    background: rgba(0,0,0,.28);
    border: 1px solid rgba(255,255,255,.18);
    color: #f1fff7;
    border-radius: 999px;
    padding: 5px 12px;
    font-size: 10px;
    font-weight: 700;
}

/* ---------------- STATISTICS PAGE ---------------- */

.mini-user { display: flex; align-items: center; gap: 10px; }
.sb-name.small   { font-size: 11px; letter-spacing: .5px; color: #e6f3ec; }
.sb-course.small { font-size: 9px; margin-top: 1px; }

.top-badge {
    background: linear-gradient(135deg, #ffe86b, #e5ae00);
    color: #00351f;
    border-radius: 999px;
    padding: 3px 9px;
    font-size: 9px;
    font-weight: 900;
    letter-spacing: 1px;
}

.bar { height: 8px; min-width: 110px; border-radius: 6px; background: rgba(255,255,255,.08); overflow: hidden; }
.bar span { display: block; height: 100%; border-radius: 6px; background: linear-gradient(90deg, #00a85a, #ffd740); }

/* ---------------- LEGEND ---------------- */

.legend { display: flex; gap: 16px; flex-wrap: wrap; font-size: 10px; color: #8da99b; margin: 0 0 14px 4px; }
.legend i { display: inline-block; width: 9px; height: 9px; border-radius: 50%; margin-right: 6px; }

/* ---------------- FOOTER ---------------- */

.footer {
    background: linear-gradient(145deg, #04200f, #021209);
    text-align: center;
    padding: 18px 12px;
    border-radius: 14px;
    border: 1px solid rgba(0,255,120,.15);
}

.footer-line {
    height: 2px;
    width: 120px;
    margin: 0 auto 12px auto;
    background: linear-gradient(90deg, transparent, #ffd740, transparent);
}

.footer-main { color: #ffe27a; font-size: 11px; font-weight: 800; letter-spacing: 3px; }
.footer-sub  { color: #7da891; font-size: 9px; letter-spacing: 2px; margin-top: 6px; }
</style>
"""))

# ============================================================
# TABLE OUTPUT
# ============================================================

table_output = widgets.Output()

# ============================================================
# HEADER
# ============================================================

header = widgets.HTML(f"""
<div class="ndmu-header">
    <div class="header-row">
        <div style="display:flex;align-items:center;gap:18px;">
            <div class="logo-wrap">
                <div class="logo-ring"></div>
                <div class="logo">NDMU<small>PORTAL</small></div>
            </div>
            <div class="school">
                <div class="school-name">NOTRE DAME OF MARBEL UNIVERSITY</div>
                <div class="school-subtitle">Academic Portal • Student Information System</div>
            </div>
        </div>
        <div class="portal-badge"><span class="live-dot"></span>STUDENT PORTAL</div>
    </div>
    <div class="portal-title">CLASS GRADE BOOK</div>
    <div class="portal-subtitle">Student Academic Management System</div>
    <div class="header-chips">
        <span class="header-chip">👥 {len(student_list)} Students</span>
        <span class="header-chip">📚 {len(df)} Subject Records</span>
    </div>
</div>
<div class="glow-line"></div>
""")

# ============================================================
# NAVIGATION BUTTONS
# ============================================================

def make_nav_button(text):
    button = widgets.Button(
        description=text,
        layout=widgets.Layout(width="140px", height="40px")
    )
    button.add_class("ndmu-nav")
    return button

home_button = make_nav_button("🏠  HOME")
grades_button = make_nav_button("📝  GRADES")
subjects_button = make_nav_button("📖  SUBJECTS")
stats_button = make_nav_button("📊  STATISTICS")

navigation = widgets.HBox(
    [home_button, grades_button, subjects_button, stats_button],
    layout=widgets.Layout(gap="10px", margin="4px 0 16px 0", flex_flow="row wrap")
)

# ============================================================
# WELCOME
# ============================================================

welcome_top = widgets.HTML("""
<div class="welcome-top">
    <div class="welcome-wave">👋</div>
    <div>
        <div class="welcome-title">Welcome to the Academic Portal</div>
        <div class="welcome-text">
            View subjects, teachers, grades, and academic information in one place.
            Click a tile below to get started.
        </div>
    </div>
</div>
""")

def make_feature_button(text):
    button = widgets.Button(
        description=text,
        layout=widgets.Layout(flex="1 1 190px", height="112px")
    )
    button.add_class("ndmu-feature")
    return button

feature_grades = make_feature_button(
    "📝  GRADES\nEvery student's records, one block each\nOpen  →"
)
feature_subjects = make_feature_button(
    "📖  SUBJECTS\nAll subjects with professors and units\nOpen  →"
)
feature_stats = make_feature_button(
    "📊  STATISTICS\nAverages, highest and lowest grades\nOpen  →"
)
feature_pdf = make_feature_button(
    "📄  PDF EXPORT\nSave the grade book in one click\nDownload  ↓"
)

feature_row = widgets.HBox(
    [feature_grades, feature_subjects, feature_stats, feature_pdf],
    layout=widgets.Layout(width="100%", gap="12px", flex_flow="row wrap", margin="18px 0 0 0")
)

welcome = widgets.VBox(
    [welcome_top, feature_row],
    layout=widgets.Layout(width="100%")
)
welcome.add_class("welcome-panel")

# ============================================================
# STUDENT INFORMATION CARD
# ============================================================

student_card = widgets.HTML(f"""
<div class="student-card">
    <div class="avatar">{html.escape(initials(student_name))}</div>
    <div>
        <div class="student-label">STUDENT INFORMATION</div>
        <div class="student-name">{html.escape(student_name)}</div>
        <div class="student-details">
            <span class="info-pill">🎓 <b>Course:</b> {html.escape(course)}</span>
            <span class="info-pill">📚 <b>Year Level:</b> {html.escape(year_level)}</span>
        </div>
    </div>
</div>
""")

# ============================================================
# STUDENT NAMES CARD  (names only)
# ============================================================

def names_card_html():
    chips = ""

    for number, label in enumerate(student_list, start=1):
        name, extra = split_student(label)
        course_html = (
            f'<span class="chip-course">• {html.escape(extra)}</span>'
            if extra else ""
        )
        chips += (
            f'<span class="name-chip">'
            f'<span class="chip-num">{number}</span>'
            f'<span class="chip-name">{html.escape(name)}</span>'
            f'{course_html}'
            f'</span>'
        )

    return f"""
    <div class="card">
        <div class="card-title">👥 Students ({len(student_list)})</div>
        <div>{chips}</div>
    </div>
    """

names_card = widgets.HTML(names_card_html())

# ============================================================
# STATISTICS CARD  (follows the students currently shown)
# ============================================================

def stat_tile(icon, number, label, accent):
    return f"""
    <div class="stat" style="--accent:{accent};">
        <div class="stat-icon">{icon}</div>
        <div>
            <div class="stat-number">{number}</div>
            <div class="stat-label">{label}</div>
        </div>
    </div>
    """

def statistics_card_html(data):
    total, average, lowest, highest = grade_stats(data)

    return f"""
    <div class="card">
        <div class="card-title">📊 Academic Overview</div>
        <div class="stat-grid">
            {stat_tile("📚", total, "TOTAL SUBJECTS", "#5af0a2")}
            {stat_tile("⭐", f"{average:.2f}", "AVERAGE GRADE", "#8be9c0")}
            {stat_tile("📉", f"{lowest:.2f}", "LOWEST GRADE", "#ffb74d")}
            {stat_tile("🏆", f"{highest:.2f}", "HIGHEST GRADE", "#ffe45c")}
        </div>
    </div>
    """

statistics_card = widgets.HTML(statistics_card_html(df))
home_stats_card = widgets.HTML(statistics_card_html(df))

# ============================================================
# SEARCH BOX + STUDENT SELECTOR
# ============================================================

ALL_STUDENTS = "ALL STUDENTS"

student_select = widgets.Dropdown(
    options=[ALL_STUDENTS] + student_list,
    value=ALL_STUDENTS,
    layout=widgets.Layout(width="290px", height="40px")
)
student_select.add_class("ndmu-input")

search_box = widgets.Text(
    placeholder="Search code, subject, description, or teacher...",
    layout=widgets.Layout(flex="1 1 260px", height="40px")
)
search_box.add_class("ndmu-input")

search_button = widgets.Button(
    description="🔍  SEARCH",
    layout=widgets.Layout(width="115px", height="40px")
)
search_button.add_class("ndmu-btn")
search_button.add_class("ndmu-btn-search")

clear_button = widgets.Button(
    description="✖  CLEAR",
    layout=widgets.Layout(width="100px", height="40px")
)
clear_button.add_class("ndmu-btn")
clear_button.add_class("ndmu-btn-clear")

pdf_button = widgets.Button(
    description="📄  SAVE AS PDF",
    layout=widgets.Layout(width="140px", height="40px")
)
pdf_button.add_class("ndmu-btn")
pdf_button.add_class("ndmu-btn-pdf")

# The dropdown is kept (it holds the selected student) but hidden;
# the clickable student pills below replace it.
student_select.layout.display = "none"

student_buttons = {}

def make_student_button(label):
    if label == ALL_STUDENTS:
        text = "✨  ALL STUDENTS"
    else:
        name, extra = split_student(label)
        text = f"{initials(name)}  ·  {name}"

    button = widgets.Button(
        description=text,
        tooltip=label,
        layout=widgets.Layout(width="auto", height="38px")
    )
    button.add_class("ndmu-pill")
    button.on_click(lambda b, chosen=label: setattr(student_select, "value", chosen))
    return button

for label in [ALL_STUDENTS] + student_list:
    student_buttons[label] = make_student_button(label)

def sync_student_buttons(*args):
    for label, button in student_buttons.items():
        if label == student_select.value:
            button.add_class("ndmu-pill-active")
        else:
            button.remove_class("ndmu-pill-active")

student_select.observe(sync_student_buttons, names="value")
sync_student_buttons()

student_filter_label = widgets.HTML('<div class="filter-label">FILTER BY STUDENT</div>')

student_pills = widgets.HBox(
    list(student_buttons.values()),
    layout=widgets.Layout(width="100%", gap="8px", flex_flow="row wrap", margin="0 0 14px 0")
)

search_area = widgets.HBox(
    [search_box, search_button, clear_button, pdf_button],
    layout=widgets.Layout(width="100%", gap="8px", flex_flow="row wrap")
)

result_chip = widgets.HTML("")

search_title = widgets.HTML(
    '<div class="card-title">🔎 Search Academic Records</div>'
)

search_section = widgets.VBox(
    [search_title, student_filter_label, student_pills, search_area, result_chip],
    layout=widgets.Layout(width="100%")
)
search_section.add_class("ndmu-panel")

legend_card = widgets.HTML("""
<div class="legend">
    <span><i style="background:#5af0a2"></i>1.00 – 1.50 Excellent</span>
    <span><i style="background:#d2f57a"></i>1.75 – 2.25 Good</span>
    <span><i style="background:#ffd54f"></i>2.50 – 3.00 Fair</span>
    <span><i style="background:#ff8a80"></i>Above 3.00 Needs attention</span>
</div>
""")

# ============================================================
# BUILD ONE STUDENT BLOCK
# ============================================================

def student_block_html(label, group):
    name, extra = split_student(label)
    total, average, lowest, highest = grade_stats(group)

    course_html = (
        f'<div class="sb-course">{html.escape(extra)}</div>' if extra else ""
    )

    rows = ""

    for number, (_, row) in enumerate(group.iterrows(), start=1):
        rows += f"""
        <tr>
            <td><span class="row-num">{number}</span></td>
            <td><span class="code-tag">{html.escape(str(row["CODE"]))}</span></td>
            <td class="desc-cell">{html.escape(str(row["DESCRIPTION"]))}</td>
            <td class="prof-cell">{html.escape(str(row["TEACHER"]))}</td>
            <td>{format_units(row["UNITS"])}</td>
            <td>{grade_pill(row["GRADE"])}</td>
        </tr>
        """

    return f"""
    <div class="card">
        <div class="student-block-header">
            <div class="avatar avatar-sm">{html.escape(initials(name))}</div>
            <div class="sb-names">
                <div class="sb-name">{html.escape(name)}</div>
                {course_html}
            </div>
            <div class="sb-pills">
                <span class="sb-pill">📚 {total} subjects</span>
                <span class="sb-pill">⭐ Avg {average:.2f}</span>
                <span class="sb-pill">🏆 {highest:.2f}</span>
                <span class="sb-pill">📉 {lowest:.2f}</span>
            </div>
        </div>
        <div class="table-wrap">
            <table class="grade-table">
                <thead>
                    <tr>
                        <th>#</th>
                        <th>CODE</th>
                        <th>DESCRIPTION</th>
                        <th>PROFESSOR</th>
                        <th>UNITS</th>
                        <th>GRADE</th>
                    </tr>
                </thead>
                <tbody>{rows}</tbody>
            </table>
        </div>
    </div>
    """

# ============================================================
# DISPLAY TABLE  (one block per student)
# ============================================================

current_data = df.copy()

def display_table(data):
    global current_data
    current_data = data.copy()

    statistics_card.value = statistics_card_html(data)

    shown_students = data["STUDENT"].nunique() if len(data) else 0
    result_chip.value = (
        f'<div class="result-chip">✨ Showing <b>{len(data)}</b> record(s) '
        f'from <b>{shown_students}</b> student(s)</div>'
    )

    with table_output:
        clear_output(wait=True)

        if len(data) == 0:
            display(HTML("""
            <div class="card">
                <div style="text-align:center;padding:25px;color:#789;">
                    🔍<br><br><b>No academic record found.</b>
                </div>
            </div>
            """))
            return

        blocks = ""

        for label, group in data.groupby("STUDENT", sort=False):
            blocks += student_block_html(label, group)

        display(HTML(blocks))

# ============================================================
# FILTER = STUDENT SELECTOR + SEARCH KEYWORD
# ============================================================

def get_filtered_data():
    data = df

    if student_select.value != ALL_STUDENTS:
        data = data[data["STUDENT"] == student_select.value]

    keyword = search_box.value.strip().lower()

    if keyword != "":
        data = data[
            data.apply(
                lambda row: keyword in " ".join(
                    str(value).lower() for value in row
                ),
                axis=1
            )
        ]

    return data

def refresh(*args):
    display_table(get_filtered_data())

student_select.observe(refresh, names="value")
search_box.observe(refresh, names="value")

# ============================================================
# SEARCH / CLEAR
# ============================================================

def search_records(button):
    refresh()

search_button.on_click(search_records)

def clear_records(button):
    search_box.value = ""
    student_select.value = ALL_STUDENTS
    refresh()

clear_button.on_click(clear_records)

# ============================================================
# PDF FUNCTION  (each student gets their own section)
# ============================================================

# Set to True to show "5 subject(s) • Average • Highest • Lowest"
# under each student's name bar in the PDF.
SHOW_STUDENT_SUMMARY_IN_PDF = False

def save_pdf(button):
    data = current_data.copy()

    if len(data) == 0:
        print("⚠️ Nothing to export. No records are being shown.")
        return

    names_in_data = list(data["STUDENT"].unique())

    if len(names_in_data) == 1:
        base_name = names_in_data[0]
    else:
        base_name = "All_Students"

    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", base_name).strip("_")
    pdf_filename = f"{safe_name}_Class_Grade_Book.pdf"

    doc = SimpleDocTemplate(
        pdf_filename,
        pagesize=landscape(A4),
        rightMargin=35,
        leftMargin=35,
        topMargin=30,
        bottomMargin=30
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "NDMUTitle",
        parent=styles["Title"],
        fontSize=20,
        leading=24,
        alignment=TA_CENTER,
        textColor=colors.HexColor("#006b3f"),
        spaceAfter=5
    )

    subtitle_style = ParagraphStyle(
        "NDMUSubtitle",
        parent=styles["Normal"],
        fontSize=10,
        leading=14,
        alignment=TA_CENTER,
        textColor=colors.HexColor("#555555"),
        spaceAfter=12
    )

    student_style = ParagraphStyle(
        "NDMUStudent",
        parent=styles["Normal"],
        fontSize=11,
        leading=16,
        alignment=TA_LEFT,
        textColor=colors.HexColor("#006b3f"),
        spaceAfter=12
    )

    name_style = ParagraphStyle(
        "StudentName",
        parent=styles["Normal"],
        fontSize=12,
        leading=16,
        textColor=colors.white,
        fontName="Helvetica-Bold"
    )

    meta_style = ParagraphStyle(
        "StudentMeta",
        parent=styles["Normal"],
        fontSize=8,
        leading=12,
        textColor=colors.HexColor("#555555"),
        spaceBefore=4,
        spaceAfter=6
    )

    story = [
        Paragraph("NOTRE DAME OF MARBEL UNIVERSITY", title_style),
        Paragraph("CLASS GRADE BOOK", title_style),
        Paragraph("Student Academic Management System", subtitle_style),
        Paragraph(
            f"<b>Prepared by:</b> {html.escape(student_name)}"
            f"&nbsp;&nbsp;&nbsp;&nbsp;<b>Course:</b> {html.escape(course)}"
            f"&nbsp;&nbsp;&nbsp;&nbsp;<b>Year Level:</b> {html.escape(year_level)}",
            student_style
        )
    ]

    # ---------- overall statistics ----------

    total_subjects, average_grade, lowest_grade, highest_grade = grade_stats(data)

    statistics_data = [
        ["Total Subjects", "Average Grade", "Lowest Grade", "Highest Grade"],
        [
            str(total_subjects),
            f"{average_grade:.2f}",
            f"{lowest_grade:.2f}",
            f"{highest_grade:.2f}"
        ]
    ]

    statistics_table = Table(
        statistics_data,
        colWidths=[170, 170, 170, 170]
    )

    statistics_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#006b3f")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("BACKGROUND", (0, 1), (-1, 1), colors.HexColor("#edf7f1")),
        ("TEXTCOLOR", (0, 1), (-1, 1), colors.HexColor("#006b3f")),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (-1, 1), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("BOX", (0, 0), (-1, -1), 1, colors.HexColor("#006b3f")),
        ("INNERGRID", (0, 0), (-1, -1), .5, colors.HexColor("#c9ddd3")),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8)
    ]))

    story.extend([statistics_table, Spacer(1, 18)])

    # ---------- one section per student ----------

    for name, group in data.groupby("STUDENT", sort=False):
        total, average, lowest, highest = grade_stats(group)

        table_data = [["#", "CODE", "DESCRIPTION", "PROFESSOR", "UNITS", "GRADE"]]

        for number, (_, row) in enumerate(group.iterrows(), start=1):
            table_data.append([
                str(number),
                str(row["CODE"]),
                str(row["DESCRIPTION"]),
                str(row["TEACHER"]),
                format_units(row["UNITS"]),
                format_grade(row["GRADE"])
            ])

        name_bar = Table(
            [[Paragraph(html.escape(str(name)), name_style)]],
            colWidths=[705]
        )
        name_bar.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#006b3f")),
            ("TOPPADDING", (0, 0), (-1, -1), 7),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
            ("LEFTPADDING", (0, 0), (-1, -1), 10)
        ]))

        grade_table = Table(
            table_data,
            colWidths=[30, 75, 260, 220, 55, 65],
            repeatRows=1
        )

        grade_table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#4f9a78")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (-1, -1), 8),
            ("ALIGN", (0, 0), (0, -1), "CENTER"),
            ("ALIGN", (-2, 0), (-1, -1), "CENTER"),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("GRID", (0, 0), (-1, -1), .5, colors.HexColor("#cccccc")),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [
                colors.white, colors.HexColor("#f1f7f4")
            ]),
            ("TEXTCOLOR", (0, 1), (-1, -1), colors.HexColor("#333333")),
            ("TOPPADDING", (0, 0), (-1, -1), 6),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 6)
        ]))

        story.extend([CondPageBreak(150), name_bar])

        if SHOW_STUDENT_SUMMARY_IN_PDF:
            story.append(
                Paragraph(
                    f"{total} subject(s) &nbsp;•&nbsp; Average {average:.2f} "
                    f"&nbsp;•&nbsp; Highest {highest:.2f} "
                    f"&nbsp;•&nbsp; Lowest {lowest:.2f}",
                    meta_style
                )
            )
        else:
            story.append(Spacer(1, 6))

        story.extend([grade_table, Spacer(1, 22)])

    story.append(
        Paragraph(
            "Generated from the NDMU Class Grade Book System",
            ParagraphStyle(
                "NDMUFooter",
                parent=styles["Normal"],
                fontSize=8,
                alignment=TA_CENTER,
                textColor=colors.HexColor("#777777")
            )
        )
    )

    doc.build(story)

    print()
    print("✅ PDF created successfully!")
    print("📄", pdf_filename)

    files.download(pdf_filename)

pdf_button.on_click(save_pdf)

# ============================================================
# PAGE CONTAINERS
# ============================================================

subjects_output = widgets.Output()
stats_output = widgets.Output()

home_box = widgets.VBox([welcome, student_card, names_card, home_stats_card])
grades_box = widgets.VBox([statistics_card, search_section, legend_card, table_output])
subjects_box = widgets.VBox([subjects_output])
stats_box = widgets.VBox([stats_output])

page_container = widgets.VBox(
    [home_box],
    layout=widgets.Layout(width="100%")
)

all_nav_buttons = [home_button, grades_button, subjects_button, stats_button]

def set_active(active_button):
    for button in all_nav_buttons:
        if button is active_button:
            button.add_class("ndmu-nav-active")
        else:
            button.remove_class("ndmu-nav-active")

# ============================================================
# SUBJECTS PAGE  (one row per subject, no grades per student)
# ============================================================

def render_subjects():
    with subjects_output:
        clear_output(wait=True)

        subjects = (
            df.groupby(
                ["CODE", "DESCRIPTION", "TEACHER", "UNITS"],
                dropna=False,
                sort=False
            )
            .agg(ENROLLED=("STUDENT", "nunique"), AVERAGE=("GRADE", "mean"))
            .reset_index()
        )

        total_units = subjects["UNITS"].sum()

        rows = ""

        for number, (_, row) in enumerate(subjects.iterrows(), start=1):
            rows += f"""
            <tr>
                <td><span class="row-num">{number}</span></td>
                <td><span class="code-tag">{html.escape(str(row["CODE"]))}</span></td>
                <td class="desc-cell">{html.escape(str(row["DESCRIPTION"]))}</td>
                <td class="prof-cell">{html.escape(str(row["TEACHER"]))}</td>
                <td>{format_units(row["UNITS"])}</td>
                <td>{int(row["ENROLLED"])}</td>
                <td>{grade_pill(row["AVERAGE"])}</td>
            </tr>
            """

        display(HTML(f"""
        <div class="card">
            <div class="card-title">📖 Subjects</div>
            <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:14px;">
                <span class="info-pill">📚 <b>{len(subjects)}</b> subject(s)</span>
                <span class="info-pill">🧮 <b>{total_units:g}</b> total units</span>
            </div>
            <div class="table-wrap">
                <table class="grade-table">
                    <thead>
                        <tr>
                            <th>#</th>
                            <th>CODE</th>
                            <th>DESCRIPTION</th>
                            <th>PROFESSOR</th>
                            <th>UNITS</th>
                            <th>STUDENTS</th>
                            <th>CLASS AVERAGE</th>
                        </tr>
                    </thead>
                    <tbody>{rows}</tbody>
                </table>
            </div>
        </div>
        """))

# ============================================================
# STATISTICS PAGE  (overall + one row per student)
# ============================================================

def render_statistics():
    with stats_output:
        clear_output(wait=True)

        entries = []

        for label, group in df.groupby("STUDENT", sort=False):
            total, average, lowest, highest = grade_stats(group)
            has_grades = group["GRADE"].notna().any()
            entries.append((label, total, average, lowest, highest, has_grades))

        graded = [round(entry[2], 2) for entry in entries if entry[5]]
        best = min(graded) if graded else None

        rows = ""

        for label, total, average, lowest, highest, has_grades in entries:
            name, extra = split_student(label)

            top = ""
            if len(entries) > 1 and has_grades and round(average, 2) == best:
                top = '<span class="top-badge">🏅 TOP</span>'

            percent = max(0, min(100, (5 - average) / 4 * 100)) if has_grades else 0

            course_html = (
                f'<div class="sb-course small">{html.escape(extra)}</div>'
                if extra else ""
            )

            rows += f"""
            <tr>
                <td>
                    <div class="mini-user">
                        <div class="avatar avatar-xs">{html.escape(initials(name))}</div>
                        <div>
                            <div class="sb-name small">{html.escape(name)}</div>
                            {course_html}
                        </div>
                        {top}
                    </div>
                </td>
                <td>{total}</td>
                <td>{grade_pill(average if has_grades else float("nan"))}</td>
                <td>{grade_pill(highest if has_grades else float("nan"))}</td>
                <td>{grade_pill(lowest if has_grades else float("nan"))}</td>
                <td><div class="bar"><span style="width:{percent:.0f}%"></span></div></td>
            </tr>
            """

        display(HTML(statistics_card_html(df)))

        display(HTML(f"""
        <div class="card">
            <div class="card-title">🏅 Statistics per Student</div>
            <div class="table-wrap">
                <table class="grade-table">
                    <thead>
                        <tr>
                            <th>STUDENT</th>
                            <th>SUBJECTS</th>
                            <th>AVERAGE</th>
                            <th>HIGHEST</th>
                            <th>LOWEST</th>
                            <th>STANDING</th>
                        </tr>
                    </thead>
                    <tbody>{rows}</tbody>
                </table>
            </div>
        </div>
        """))

# ============================================================
# NAVIGATION ACTIONS
# ============================================================

def show_home(button=None):
    set_active(home_button)
    page_container.children = [home_box]

def show_grades(button=None):
    set_active(grades_button)
    page_container.children = [grades_box]

def show_subjects(button=None):
    set_active(subjects_button)
    render_subjects()
    page_container.children = [subjects_box]

def show_statistics(button=None):
    set_active(stats_button)
    render_statistics()
    page_container.children = [stats_box]

home_button.on_click(show_home)
grades_button.on_click(show_grades)
subjects_button.on_click(show_subjects)
stats_button.on_click(show_statistics)

# Home page tiles do the same as the navigation buttons
feature_grades.on_click(show_grades)
feature_subjects.on_click(show_subjects)
feature_stats.on_click(show_statistics)
feature_pdf.on_click(save_pdf)

# ============================================================
# FOOTER
# ============================================================

footer = widgets.HTML("""
<div class="footer">
    <div class="footer-line"></div>
    <div class="footer-main">NOTRE DAME OF MARBEL UNIVERSITY STYLE</div>
    <div class="footer-sub">CLASS GRADE BOOK • STUDENT ACADEMIC MANAGEMENT SYSTEM</div>
</div>
""")

# ============================================================
# MAIN APPLICATION
# ============================================================

app = widgets.VBox(
    [
        header,
        navigation,
        page_container,
        footer
    ],
    layout=widgets.Layout(
        width="100%",
        max_width="1200px",
        margin="0 auto"
    )
)

# ============================================================
# DISPLAY
# ============================================================

display(app)
display_table(df)
set_active(home_button)


✅ PDF created successfully!
📄 All_Students_Class_Grade_Book.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ PDF created successfully!
📄 ANGELA_REYES_-_INFORMATION_TECHNOLOGY_Class_Grade_Book.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ PDF created successfully!
📄 PETER_GONZALES_-_BIOLOGY_Class_Grade_Book.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>